# Skill Training Example

This notebook allows you to:
1. Connect to the R2 Robot backend, which includes a training endpoint.
2. Start training jobs.
3. Monitor training progress.
4. Cancel training if needed.

Uses the `TrainerClient` from the main SDK client module (`r2_labs.sdk.client`).

Remember to start the R2 backend first before running this notebook.

In [ ]:
import time
from IPython.display import clear_output, display
import matplotlib.pyplot as plt

from r2_labs.rpc import client as rpc_client
from r2_labs.sdk import client as sdk_client
from r2_labs.sdk import rpc_api

%load_ext autoreload
%autoreload 2


## 1. Configure Server Connection

Set the hostname/IP of the machine running the R2 backend.

In [ ]:
# Configure the server address
TRAINING_SERVER_HOST = "localhost"  # Change to your server IP/hostname
TRAINING_SERVER_PORT = rpc_api.DEFAULT_MODEL_TRAINER_PORT  # 7534

server_address = f"tcp://{TRAINING_SERVER_HOST}:{TRAINING_SERVER_PORT}"
print(f"Training server: {server_address}")

## 2. Connect to the Server

Create a `TrainerClient` using the SDK client module.

In [ ]:
# Create the base RPC client
base_client = rpc_client.BaseClient(
    server_address,
    timeout=30000,  # 30 second timeout
)

# Create the trainer client from the SDK
trainer = sdk_client.TrainerClient(base_client)

# Test connection by getting status
try:
    status = trainer.get_training_status()
    print("Connected to Training server!")
except Exception as e:
    print(f"Failed to connect: {e}")

## 4. Start Training

### Option A: Use entry_filter (auto-build the dataset from the R2 Data Warehouse)

In [ ]:
# Training configuration with entry filter
MODEL_NAME = "rectify_open_latch"
TRAINING_STEPS = 40000
ENTRY_FILTER = "rectify_open_latch*"  # Glob pattern to match data warehouse entries

# Start training with entry_filter
response = trainer.train_skill_model(
    model_name=MODEL_NAME,
    training_steps=TRAINING_STEPS,
    entry_filter=ENTRY_FILTER,
    batch_size=32,
    prediction_horizon=32,
    force_rebuild=False,  # Set to True to force rebuild the dataset
    timeout=600000,  # 10 minute timeout to give time for dataset rebuild.
)

if response.error:
    print(f"ERROR: {response.error}")
else:
    print("Training process started!")
    print("Use get_training_status() or monitor_training() to track progress.")

## 3. Check Current Training Status

Use this to check if training is already running.

In [ ]:
def print_status(status):
    """Pretty print training status."""
    print("=" * 50)
    phase = getattr(status, 'phase', 'unknown')
    print(f"Phase: {phase}")
    print(f"Is Finished: {status.is_finished}")
    if phase == "exporting":
        export_processed = getattr(status, 'export_entries_processed', 0)
        export_total = getattr(status, 'export_entries_total', 0)
        print(f"Export Progress: {export_processed} / {export_total}")
    else:
        print(f"Steps: {status.steps_completed} / {status.max_steps}")
        if status.loss is not None:
            print(f"Loss: {status.loss:.6f}")
        if status.fps is not None:
            print(f"Speed: {status.fps:.2f} steps/sec")
        if status.seconds_per_step is not None:
            print(f"Time per step: {status.seconds_per_step:.3f} sec")
        if status.metrics:
            print("Metrics:", status.metrics)
    print("=" * 50)

# Check current status
status = trainer.get_training_status()
print_status(status)

## 5. Monitor Training Progress

This cell will continuously monitor and display training progress with a live loss plot.

In [ ]:
def monitor_training(trainer: sdk_client.TrainerClient, poll_interval=5, max_history=1000):
    """Monitor training progress with live updates.

    Handles both dataset export phase and training phase.
    """
    steps_history = []
    loss_history = []
    fps_history = []
    export_history = []

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    try:
        while True:
            status = trainer.get_training_status()

            if status.is_finished:
                print("\nTraining finished!")
                break

            clear_output(wait=True)

            # Determine phase from status (use getattr for backward compatibility)
            phase = getattr(status, 'phase', 'training')
            export_processed = getattr(status, 'export_entries_processed', 0)
            export_total = getattr(status, 'export_entries_total', 0)

            if phase == "exporting":
                # Show export progress
                progress = export_processed / max(export_total, 1) * 100
                print(f"Phase: Dataset Export")
                print(f"Entries: {export_processed:,} / {export_total:,} ({progress:.1f}%)")

                # Track export history
                export_history.append(export_processed)

                # Update export visualization
                ax1.clear()
                ax2.clear()

                if len(export_history) > 1:
                    ax1.plot(export_history, 'b-', linewidth=1)
                    ax1.axhline(y=export_total, color='r', linestyle='--', label='Target')
                    ax1.set_xlabel('Polls')
                    ax1.set_ylabel('Entries Processed')
                    ax1.set_title('Dataset Export Progress')
                    ax1.grid(True, alpha=0.3)
                    ax1.legend()

                # Progress bar visualization
                ax2.barh([0], [export_processed], color='blue', label='Processed')
                ax2.barh([0], [export_total - export_processed], left=[export_processed],
                        color='lightgray', label='Remaining')
                ax2.set_xlim(0, max(export_total, 1))
                ax2.set_yticks([])
                ax2.set_xlabel('Entries')
                ax2.set_title(f'Export Progress: {progress:.1f}%')
                ax2.legend()

            elif phase == "training":
                # Show training progress
                if status.loss is not None:
                    steps_history.append(status.steps_completed)
                    loss_history.append(status.loss)
                if status.fps is not None:
                    fps_history.append(status.fps)

                # Trim history if needed
                if len(steps_history) > max_history:
                    steps_history = steps_history[-max_history:]
                    loss_history = loss_history[-max_history:]
                if len(fps_history) > max_history:
                    fps_history = fps_history[-max_history:]

                progress = status.steps_completed / status.max_steps * 100 if status.max_steps > 0 else 0
                print(f"Phase: Training")
                print(f"Step: {status.steps_completed:,} / {status.max_steps:,} ({progress:.1f}%)")
                if status.loss is not None:
                    print(f"Loss: {status.loss:.6f}")
                if status.fps is not None:
                    print(f"Speed: {status.fps:.2f} steps/sec")
                    if status.max_steps > status.steps_completed:
                        eta_seconds = (status.max_steps - status.steps_completed) / (status.fps + 1e-8)
                        eta_minutes = eta_seconds / 60
                        if eta_minutes > 60:
                            print(f"ETA: {eta_minutes/60:.1f} hours")
                        else:
                            print(f"ETA: {eta_minutes:.1f} minutes")

                # Update training plots
                ax1.clear()
                ax2.clear()

                if len(loss_history) > 1:
                    ax1.plot(steps_history, loss_history, 'b-', linewidth=1)
                    ax1.set_xlabel('Steps')
                    ax1.set_ylabel('Loss')
                    ax1.set_title('Training Loss')
                    ax1.grid(True, alpha=0.3)
                    ax1.set_yscale('log')

                if len(fps_history) > 1:
                    ax2.plot(fps_history[-100:], 'g-', linewidth=1)
                    ax2.set_xlabel('Recent Samples')
                    ax2.set_ylabel('Steps/sec')
                    ax2.set_title('Training Speed (FPS)')
                    ax2.grid(True, alpha=0.3)

            else:
                print(f"Phase: {phase}")
                print("Waiting...")

            plt.tight_layout()
            display(fig)
            time.sleep(poll_interval)

    except KeyboardInterrupt:
        print("\nMonitoring stopped by user.")
    finally:
        plt.close(fig)

    return steps_history, loss_history

# Start monitoring (Ctrl+C or interrupt kernel to stop)
steps, losses = monitor_training(trainer, poll_interval=5)

## 6. Cancel Training

Use this to stop training early if needed.

In [ ]:
# Cancel training (set export_model=True to export before cancelling)
response = trainer.cancel_training(export_model=False)

if response.success:
    print("Training cancelled successfully!")
    if response.model_id:
        print(f"Model ID: {response.model_id}")
else:
    print(f"Failed to cancel: {response.error}")

## 7. Export Model

Export the trained model to the model warehouse.

In [ ]:
# Export model after training
response = trainer.export_model()

if response.success:
    print("Model exported successfully!")
    print(f"Model ID: {response.model_id}")
else:
    print(f"Failed to export model: {response.error}")

## 8. One-time Status Check

In [ ]:
# Quick status check
status = trainer.get_training_status()
print_status(status)